In [4]:
import argparse
from itertools import product

import numpy as np

from cc2cc import add_args, cc, ucc
from cc2cc.utils import Grid, gen_mole, print_computer_info
from cc2cc.utils.env_var import DATA_PATH
from cc2cc.utils.parser import gen_name_args

origin_mol_str_list = [
    # "molecule0-W4_11",
    # "molecule1-W4_11",
    # "molecule2-W4_11",
    # "molecule3-W4_11",
    # "molecule4-W4_11",
    "molecule5-W4_11",
]
name_mol_str_list = [
    # "molecule0",
    # "molecule1",
    # "molecule2",
    # "molecule3",
    # "molecule4",
    "molecule5",
    # "molecule6",
]
name_mol_str_exclude_list = [
]

mol_elements_dict = {}
len_elements_dict = {}

if __name__ == "__main__":
    error_molecule = []

    name_mol_list = gen_name_args(name_mol_str_list, "gmtkn-def2")
    name_mol_exclude_list = gen_name_args(
        name_mol_str_exclude_list, "gmtkn-def2", if_exclude=True
    )
    name_mol_list = [mol for mol in name_mol_list if mol not in name_mol_exclude_list]

    origin_mol_list = gen_name_args(origin_mol_str_list, "gmtkn-def2")
    name_mol_list = origin_mol_list + name_mol_list

    error_molecule = []
    print(f"Name Molecule List: {name_mol_list}")

    for name_mol in name_mol_list:
        try:
            mol = gen_mole(
                name_mol,
                0,
                1,
                0,
                "def2-TZVPD",
                "gmtkn-def2",
                if_rotate=True,
                if_rotate_random=False,
                solve_symmetry=True,
                verbose=1,
            )

            mol_elements = list(np.array(mol.elements))
            mol_atom_coords = list(mol.atom_coords())
            mol_atom_coords.append(name_mol)
            # print(mol_atom_coords)
            mol_elements.extend([mol.charge, mol.spin])
            mol_elements_str = "-".join(map(str, mol_elements))
            if mol_elements_str not in mol_elements_dict:
                mol_elements_dict[mol_elements_str] = [mol_atom_coords]
            else:
                mol_elements_dict[mol_elements_str].append(mol_atom_coords)
            len_elements_dict[mol_elements_str] = len(list(np.array(mol.elements)))

        except (ValueError, RuntimeError) as e:
            print(f"ERROR: {name_mol}")
            print(e)
            error_molecule.append(name_mol)
            print(f"Error molecule: {error_molecule}")
        finally:
            print(f"Processed: {name_mol}")
        print()

    print(f"Error molecule: {error_molecule}")

Name Molecule List: ['W4_11-cf4', 'W4_11-sif4', 'ALK8-li4_c', 'G2RC-61', 'G2RC-62', 'G2RC-66', 'G2RC-67', 'HAL59-29_CF3Br-benB', 'HAL59-30_CF3I-benB', 'HAL59-F3CI', 'IL16-152B', 'IL16-214B', 'IL16-229B', 'W4_11-cf4', 'W4_11-sif4', 'HAL59-BrBr_FCCH', 'HAL59-FI_FCCH', 'PNICO23-22b', 'DC13-o3_c2h2_add', 'RSE43-P5', 'RSE43-P7', 'TAUT15-7a', 'TAUT15-7b', 'YBDE18-nf3-ch2', 'YBDE18-pf3-ch2', 'IL16-230B', 'RSE43-E5', 'RSE43-E7', 'BHPERI-04r', 'BHPERI-09r', 'BHPERI-13ts_1a', 'DARC-furane', 'DC13-o3_c2h4_add', 'HAL59-MeI_FCCH', 'RC21-7p1', 'RSE43-P42', 'TAUT15-2a', 'TAUT15-2b', 'AHB21-19', 'BHPERI-05r', 'BHPERI-07r', 'BHPERI-13ts_2a', 'BHPERI-13ts_4a', 'PA26-gly', 'PA26-phosphapyrrol', 'PNICO23-20', 'PX13-hf_5', 'PX13-hf_5_ts', 'RC21-7p5', 'RSE43-E42', 'RSE43-P17', 'RSE43-P31', 'RSE43-P38', 'RSE43-P41', 'RSE43-P6', 'AHB21-18', 'BHDIV10-ed6', 'BHDIV10-ts6', 'BHPERI-02r', 'BHPERI-03r', 'BHPERI-1,3-Cyclopentadiene', 'BHPERI-13ts_3a', 'BHPERI-13ts_5a', 'BHPERI-TS5', 'CDIE20-P20', 'CDIE20-R20', 'CDIE

In [7]:
for mol_elements_name, mol_elements in mol_elements_dict.items():
    print(f"Processing {len_elements_dict[mol_elements_name]}")
    # print(f"{mol_elements}")

    identifiables = [0]
    for i_elements in range(1, len(mol_elements)):
        distance_list = np.zeros(len(identifiables))
        for iter, identifiable in enumerate(identifiables):
            for i_element in range(len(mol_elements[i_elements]) - 1):
                distance_list[iter] = max(
                    np.linalg.norm(
                        mol_elements[i_elements][i_element]
                        - mol_elements[identifiable][i_element]
                    ),
                    distance_list[iter],
                )
        if np.all(distance_list > 0.2):
            identifiables.append(i_elements)
        # else:
        #     print(f"Skipping {mol_elements[i_elements][-1]}")

    # print("===identifiables===")
    for identifiable in identifiables:
        if mol_elements[identifiable][-1].startswith("W4_11"):
            continue
        print(f'"{mol_elements[identifiable][-1]}",')
        # print(
        #     f"{np.array2string(np.array(mol_elements[identifiable][:-1]), formatter={'float': '{: .2f}'.format})}"
        # )
    print()

Processing 5

Processing 5

Processing 5
"ALK8-li4_c",

Processing 5
"G2RC-62",

Processing 5
"G2RC-67",

Processing 5
"HAL59-29_CF3Br-benB",

Processing 5
"HAL59-30_CF3I-benB",

Processing 5
"IL16-152B",
"IL16-214B",
"IL16-229B",

Processing 6
"HAL59-BrBr_FCCH",

Processing 6
"HAL59-FI_FCCH",

Processing 6
"PNICO23-22b",

Processing 7
"DC13-o3_c2h2_add",

Processing 7
"RSE43-P5",

Processing 7
"RSE43-P7",

Processing 7
"TAUT15-7a",
"TAUT15-7b",

Processing 7
"YBDE18-nf3-ch2",

Processing 7
"YBDE18-pf3-ch2",

Processing 8
"IL16-230B",

Processing 8
"RSE43-E5",

Processing 8
"RSE43-E7",

Processing 9
"BHPERI-04r",

Processing 9
"BHPERI-09r",

Processing 9
"BHPERI-13ts_1a",

Processing 9
"DC13-o3_c2h4_add",

Processing 9
"HAL59-MeI_FCCH",

Processing 9
"RC21-7p1",

Processing 9
"RSE43-P42",

Processing 9
"TAUT15-2a",
"TAUT15-2b",

Processing 10
"AHB21-19",

Processing 10
"BHPERI-05r",

Processing 10
"BHPERI-07r",
"PA26-phosphapyrrol",

Processing 10
"BHPERI-13ts_2a",

Processing 10
"BHPE

In [8]:
mol_elements_dict

{'Be-Cl-Cl-0-0': [[array([0., 0., 0.]),
   array([-3.40120467,  0.        ,  0.        ]),
   array([3.40120467, 0.        , 0.        ]),
   'W4_11-becl2']],
 'Be-F-F-0-0': [[array([0., 0., 0.]),
   array([-2.60704726,  0.        ,  0.        ]),
   array([2.60704726, 0.        , 0.        ]),
   'W4_11-bef2']],
 'C-Cl-Cl-0-0': [[array([0.       , 1.6089375, 0.       ]),
   array([-2.64638569, -0.27252296,  0.        ]),
   array([ 2.64638569, -0.27252296,  0.        ]),
   'W4_11-ccl2']],
 'C-F-F-0-0': [[array([ 0.        , -1.13580973,  0.        ]),
   array([-1.94261956,  0.35899387,  0.        ]),
   array([1.94261956, 0.35899387, 0.        ]),
   'W4_11-cf2']],
 'O-Cl-Cl-0-0': [[array([ 0.        , -1.48572494,  0.        ]),
   array([-2.64783133,  0.33525673,  0.        ]),
   array([2.64783133, 0.33525673, 0.        ]),
   'W4_11-cl2o']],
 'C-N-Cl-0-0': [[array([1.28237933, 0.        , 0.        ]),
   array([3.47820707, 0.        , 0.        ]),
   array([-1.80865703,  0.   